<a href="https://colab.research.google.com/github/Prativa239-pd/PRATIVA_RECENT/blob/main/Rice_4States_Satellite_Figures_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Satellite figures for the paper: study area, every state and every district
**Odisha, Chhattisgarh, Punjab, West Bengal - Sentinel-2 / Sentinel-1, Kharif 2020 (Google Earth Engine, Colab)**

This notebook makes journal-style figures (PNG, 300 dpi, lon/lat axes, scale bar, north arrow):

* **Fig. 1** - study area: the four states with their district boundaries
* **One figure per state** - district map with names, Sentinel-2 true colour, NIR temporal composite, VH temporal composite
* **One figure per district** - Sentinel-2 true colour, NIR temporal composite, VH temporal composite



In [ ]:
# CELL 1 - Earth Engine set-up
import re, datetime
import ee
import pandas as pd
PROJECT = 'my-project-50729-prativa'

try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()          # opens the Google sign-in the first time
    ee.Initialize(project=PROJECT)

print('Earth Engine initialised with project:', PROJECT)

Earth Engine initialised with project: my-project-50729-prativa


In [ ]:
# CELL 2 - CONFIGURATION (the only cell you normally edit)

STATES = ['Odisha', 'Chhattisgarh', 'Punjab', 'West Bengal']
ALIASES = {
    'Odisha':       ['odisha', 'orissa'],
    'Chhattisgarh': ['chhattisgarh', 'chattisgarh', 'chhatisgarh'],
    'Punjab':       ['punjab'],
    'West Bengal':  ['west bengal'],
}

YEAR = 2020
STAGE_WINDOWS = {
    'B': ('2020-06-01', '2020-07-15'),   # Beginning -> red channel
    'M': ('2020-07-16', '2020-09-15'),   # Middle    -> green channel
    'E': ('2020-09-16', '2020-10-31'),   # Ending    -> blue channel
}
STATE_WINDOWS = {s: STAGE_WINDOWS for s in STATES}

CLOUDY_PIXEL_MAX = 20     # Sentinel-2 scenes with >= 20 % cloudy pixels are dropped (thesis Sec. 3.2.2)
USE_SCL_MASK = True       # also mask cloud / shadow / cirrus / snow pixels with the SCL band.
                          # Set False to reproduce "median only" exactly as in the thesis text.

SCALE = 10                # metres
CRS = 'EPSG:4326'
NODATA = -9999.0          # value written where there is no data / outside the district
NORM = {'VH': (-25.0, -5.0), 'VV': (-20.0, 0.0), 'SWIR': (0.0, 0.5)}

ALL_SIGNALS = ['NIR', 'SWIR', 'RED', 'VV', 'VH', 'VHVV', 'VH_SWIR', 'VV_SWIR']
SIGNALS_TO_EXPORT = ALL_SIGNALS            # e.g. ['NIR'] to export one composition only
STATES_TO_RUN = STATES                     # e.g. ['Chhattisgarh']
DISTRICT_FILTER = None                     # e.g. ['Raipur', 'Durg'] (exact spelling as printed below)

DISTRICT_ASSET = None                      # e.g. 'projects/<project>/assets/india_districts'
CUSTOM_FIELDS = {'state': 'STATE', 'district': 'DISTRICT', 'code': None}   # used only with DISTRICT_ASSET

DRIVE_FOLDER = 'rice_compositions_kharif2020'
DRY_RUN = True                             # True = only count tasks; False = really start exports
CHECK_DISTRICT_COUNTS = True               # skip a district/stage that has no S1 or S2 scene

EXPECTED_AREA_KM2 = {'Odisha': 155707, 'Chhattisgarh': 135191, 'Punjab': 50360, 'West Bengal': 88752}

In [ ]:
# CELL 3 - State and district boundaries
def norm_name(s):
    return re.sub(r'\s+', ' ', str(s)).strip().lower()

if DISTRICT_ASSET:
    SRC = ee.FeatureCollection(DISTRICT_ASSET)
    F_STATE, F_DIST, F_CODE = CUSTOM_FIELDS['state'], CUSTOM_FIELDS['district'], CUSTOM_FIELDS['code']
else:
    SRC = (ee.FeatureCollection('FAO/GAUL/2015/level2')
           .filter(ee.Filter.eq('ADM0_NAME', 'India')))
    F_STATE, F_DIST, F_CODE = 'ADM1_NAME', 'ADM2_NAME', 'ADM2_CODE'

all_state_names = SRC.aggregate_array(F_STATE).distinct().getInfo()
print('State names present in the boundary layer:', sorted(str(n) for n in all_state_names))

def resolve_state(state):
    wanted = ALIASES[state]
    hits = [n for n in all_state_names if n is not None and norm_name(n) in wanted]
    if len(hits) != 1:
        raise ValueError(f'Could not match {state!r} uniquely. Matches: {hits}. '
                         f'Check ALIASES against the printed state names.')
    return hits[0]

STATE_FC, DIST_ROWS, STATE_AREA = {}, {}, {}
for st in STATES_TO_RUN:
    full = SRC.filter(ee.Filter.eq(F_STATE, resolve_state(st))).sort(F_DIST)
    STATE_AREA[st] = full.geometry(100).area(100).divide(1e6).getInfo()
    fc = full if not DISTRICT_FILTER else full.filter(ee.Filter.inList(F_DIST, DISTRICT_FILTER))
    props = [F_DIST] + ([F_CODE] if F_CODE else [])
    feats = fc.map(lambda f: ee.Feature(
        None, f.toDictionary(props).set('area_km2', f.geometry().area(100).divide(1e6)))
    ).getInfo()['features']
    if not feats:
        raise ValueError(f'No districts found for {st}. Check DISTRICT_FILTER / DISTRICT_ASSET fields.')
    DIST_ROWS[st] = [{
        'i': i,
        'district': r['properties'][F_DIST],
        'code': (r['properties'].get(F_CODE) if F_CODE else None),
        'area_km2': round(r['properties']['area_km2'], 1),
    } for i, r in enumerate(feats)]
    STATE_FC[st] = fc

for st in STATES_TO_RUN:
    exp = EXPECTED_AREA_KM2.get(st)
    flag = ''
    if exp and abs(STATE_AREA[st] - exp) / exp > 0.10:
        flag = '   <-- differs >10 % from thesis Table 3.7, check the boundary'
    print(f'\n=== {st}: {len(DIST_ROWS[st])} districts in this layer, '
          f'state area {STATE_AREA[st]:,.0f} km2 (thesis: {exp}){flag}')
    print(pd.DataFrame(DIST_ROWS[st]).drop(columns='i').to_string(index=False))

/usr/local/lib/python3.13/dist-packages/ee/deprecation.py:215: DeprecationWarning: 

Attention required for FAO/GAUL/2015/level2! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by FAO/GAUL/2025/level2

Learn more: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_2015_level2

  warnings.warn(warning, category=DeprecationWarning)


State names present in the boundary layer: ['Andaman and Nicobar', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra and Nagar Haveli', 'Daman and Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jharkhand', 'Karnataka', 'Kerala', 'Lakshadweep', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Orissa', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']

=== Odisha: 30 districts in this layer, state area 156,075 km2 (thesis: 155707)
     district  code  area_km2
        Angul 70206    6391.8
    Baleshwar 70207    3697.1
     Baragarh 70208    5821.7
      Bhadrak 70209    2454.3
     Bolangir 70210    6595.8
        Boudh 70211    3132.7
      Cuttack 17809    3955.6
      Deogarh 70212    2821.6
    Dhenkanal 17810    4493.8
     Gajapati 70213    4080.4
       Ganjam 17811    8417.7
Jagatsinghpur 70214    1691.3
       Jajpur 

In [ ]:
# CELL 4 - Compositions (S2 + S1 -> stage medians -> 3-band temporal composites)
def window_dates(win):
    start, end_inclusive = win
    return ee.Date(start), ee.Date(end_inclusive).advance(1, 'day')   # filterDate end is exclusive

def prep_s2(img):
    # B4/B8/B11 -> RED/NIR/SWIR reflectance (0-1); B11 (20 m) bilinearly resampled to 10 m
    refl = (img.select(['B4', 'B8', 'B11'], ['RED', 'NIR', 'SWIR'])
            .resample('bilinear').divide(10000))
    if USE_SCL_MASK:
        scl = img.select('SCL')
        bad = (scl.eq(0).Or(scl.eq(1)).Or(scl.eq(3)).Or(scl.eq(8))
               .Or(scl.eq(9)).Or(scl.eq(10)).Or(scl.eq(11)))   # no data, saturated, shadow, cloud, cirrus, snow
        refl = refl.updateMask(bad.Not())
    return refl

def s2_collection(geom, win):
    start, end = window_dates(win)
    return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(geom).filterDate(start, end)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUDY_PIXEL_MAX)))

def s1_collection(geom, win):
    start, end = window_dates(win)
    pol = 'transmitterReceiverPolarisation'
    return (ee.ImageCollection('COPERNICUS/S1_GRD')
            .filterBounds(geom).filterDate(start, end)
            .filter(ee.Filter.eq('instrumentMode', 'IW'))
            .filter(ee.Filter.listContains(pol, 'VV'))
            .filter(ee.Filter.listContains(pol, 'VH'))
            .select(['VV', 'VH']))                              # already in dB in Earth Engine

def stage_image(geom, win):
    s2 = s2_collection(geom, win).map(prep_s2).median()          # bands RED, NIR, SWIR
    s1 = s1_collection(geom, win).median()                       # bands VV, VH
    vhvv = s1.select('VH').subtract(s1.select('VV')).rename('VHVV')   # from the stage medians (thesis Sec. 3.2.1)
    return ee.Image.cat([s2, s1, vhvv])

def unit(img, lo, hi):
    return img.subtract(lo).divide(hi - lo).clamp(0, 1)

def signal_band(stage_img, sig):
    if sig in ('NIR', 'SWIR', 'RED', 'VV', 'VH', 'VHVV'):
        return stage_img.select(sig)
    if sig == 'VH_SWIR':
        return unit(stage_img.select('VH'), *NORM['VH']).add(unit(stage_img.select('SWIR'), *NORM['SWIR']))
    if sig == 'VV_SWIR':
        return unit(stage_img.select('VV'), *NORM['VV']).add(unit(stage_img.select('SWIR'), *NORM['SWIR']))
    raise ValueError(f'Unknown signal {sig}')

def build_compositions(geom, windows, signals):
    stages = {t: stage_image(geom, windows[t]) for t in ('B', 'M', 'E')}
    comps = {}
    for sig in signals:
        # band order B, M, E = R, G, B of the false-colour composite (Eq. 3.1)
        comps[sig] = ee.Image.cat(
            [signal_band(stages[t], sig).rename(f'{sig}_{t}') for t in ('B', 'M', 'E')]).toFloat()
    return comps

def scene_counts(geom, windows):
    d = {}
    for t in ('B', 'M', 'E'):
        d[f'S2_{t}'] = s2_collection(geom, windows[t]).size()
        d[f'S1_{t}'] = s1_collection(geom, windows[t]).size()
    return ee.Dictionary(d)

In [ ]:
# CELL 5 - FIGURE CONFIGURATION
import os, io, time, math, shutil, textwrap
import requests
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.patheffects as pe
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator, FuncFormatter

OUT_DIR = 'figures_kharif2020'

# Panel keys: 'index' (state figures only: district map with names), 'truecolour',
# or any composition name in ALL_SIGNALS: NIR, SWIR, RED, VV, VH, VHVV, VH_SWIR, VV_SWIR
PANELS_STATE = ['index', 'truecolour', 'NIR', 'VH']
PANELS_DISTRICT = ['truecolour', 'NIR', 'VH']

MAKE_STUDY_AREA_FIGURE = True
MAKE_STATE_FIGURES = True
MAKE_DISTRICT_FIGURES = True

STATE_DIM = 1800        # longest side of each state panel image, pixels
DISTRICT_DIM = 1400     # longest side of each district panel image, pixels
DPI = 300
SAVE_PDF = False        # also save a PDF copy of every figure
SKIP_EXISTING = True    # lets you re-run the cell and continue where it stopped
LABEL_FONT = 5          # district-name font size on the state maps
SAVE_TO_DRIVE = False   # True: copy all figures to your Google Drive at the end

PANEL_TITLE = {
    'index': 'District map',
    'truecolour': 'Sentinel-2 true colour (median of the season)',
    'NIR': 'Sentinel-2 NIR (B8) temporal composite',
    'SWIR': 'Sentinel-2 SWIR (B11) temporal composite',
    'RED': 'Sentinel-2 red (B4) temporal composite',
    'VV': 'Sentinel-1 VV temporal composite',
    'VH': 'Sentinel-1 VH temporal composite',
    'VHVV': 'Sentinel-1 VH-VV temporal composite',
    'VH_SWIR': 'VH + SWIR fused temporal composite',
    'VV_SWIR': 'VV + SWIR fused temporal composite',
}
PANEL_VIS = {'NIR': (0.02, 0.45), 'SWIR': (0.02, 0.40), 'RED': (0.0, 0.2), 'VV': (-20, -5),
             'VH': (-25, -10), 'VHVV': (-15, 0), 'VH_SWIR': (0.0, 2.0), 'VV_SWIR': (0.0, 2.0)}
STATE_COLOURS = ['#e41a1c', '#377eb8', '#4daf4a', '#ff7f00']
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# CELL 6 - Helper functions for the figures
def scl_bad(img):
    scl = img.select('SCL')
    return (scl.eq(0).Or(scl.eq(1)).Or(scl.eq(3)).Or(scl.eq(8))
            .Or(scl.eq(9)).Or(scl.eq(10)).Or(scl.eq(11)))

def prep_rgb(img):
    rgb = img.select(['B4', 'B3', 'B2']).divide(10000)
    if USE_SCL_MASK:
        rgb = rgb.updateMask(scl_bad(img).Not())
    return rgb

def outline_layer(fc, width=1, colour='ffffff'):
    return (ee.Image().byte().paint(featureCollection=fc, color=1, width=width)
            .visualize(min=0, max=1, palette=[colour, colour]))

def panel_image(panel, geom, fc, windows):
    # returns an already-visualised 3-band image (boundaries blended on top)
    if panel == 'index':
        fill = (ee.Image().byte().paint(featureCollection=fc, color=1)
                .visualize(min=0, max=1, palette=['#e6e6e6', '#e6e6e6']))
        return fill.blend(outline_layer(fc, 1, '555555'))
    if panel == 'truecolour':
        start = ee.Date(windows['B'][0])
        end = ee.Date(windows['E'][1]).advance(1, 'day')
        col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
               .filterBounds(geom).filterDate(start, end)
               .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUDY_PIXEL_MAX))
               .map(prep_rgb))
        img = col.median().clip(geom).visualize(bands=['B4', 'B3', 'B2'], min=0.02, max=0.22, gamma=1.2)
    else:
        comp = build_compositions(geom, windows, [panel])[panel].clip(geom)
        lo, hi = PANEL_VIS[panel]
        img = comp.visualize(bands=[f'{panel}_B', f'{panel}_M', f'{panel}_E'], min=lo, max=hi)
    return img.blend(outline_layer(fc, 1, 'ffffff'))

def bbox_of(geom):
    ring = geom.bounds().getInfo()['coordinates'][0]
    xs, ys = [p[0] for p in ring], [p[1] for p in ring]
    return min(xs), min(ys), max(xs), max(ys)

def fetch_png(img, geom, dim):
    last = None
    for d in (dim, int(dim * 0.6)):          # retry once at lower resolution if EE times out
        try:
            url = img.getThumbURL({'region': geom.bounds(), 'dimensions': d,
                                   'crs': 'EPSG:4326', 'format': 'png'})
            r = requests.get(url, timeout=900)
            if r.status_code != 200 or not r.headers.get('Content-Type', '').startswith('image'):
                raise RuntimeError(r.text[:300])
            return mpimg.imread(io.BytesIO(r.content), format='png')
        except Exception as e:
            last = e
    raise last

def window_text(w):
    import datetime as _dt
    f = lambda s: _dt.datetime.strptime(s, '%Y-%m-%d').strftime('%d %b')
    return f'{f(w[0])}-{f(w[1])}'

HALO = [pe.withStroke(linewidth=1.5, foreground='white')]

def draw_scalebar_north(ax, bb):
    x0, y0, x1, y1 = bb
    dx, dy = x1 - x0, y1 - y0
    km_per_deg = 111.32 * math.cos(math.radians((y0 + y1) / 2))
    width_km = dx * km_per_deg
    nice = [1, 2, 5, 10, 20, 25, 50, 100, 200, 250, 500]
    cand = [v for v in nice if v <= 0.25 * width_km]
    L = cand[-1] if cand else 1
    xs, ys = x0 + 0.05 * dx, y0 + 0.05 * dy
    xe = xs + L / km_per_deg
    ax.plot([xs, xe], [ys, ys], color='white', lw=5, solid_capstyle='butt', zorder=5)
    ax.plot([xs, xe], [ys, ys], color='black', lw=3, solid_capstyle='butt', zorder=6)
    ax.text((xs + xe) / 2, ys + 0.02 * dy, f'{L} km', ha='center', va='bottom',
            fontsize=6, path_effects=HALO, zorder=7)
    nx = x1 - 0.07 * dx
    ax.annotate('', xy=(nx, y1 - 0.05 * dy), xytext=(nx, y1 - 0.15 * dy),
                arrowprops=dict(arrowstyle='-|>', color='black', lw=1.2), zorder=7)
    ax.text(nx, y1 - 0.045 * dy, 'N', ha='center', va='bottom', fontsize=7,
            fontweight='bold', path_effects=HALO, zorder=7)

def format_axes(ax, bb):
    x0, y0, x1, y1 = bb
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
    ax.set_aspect(1 / math.cos(math.radians((y0 + y1) / 2)))
    ax.xaxis.set_major_locator(MaxNLocator(4)); ax.yaxis.set_major_locator(MaxNLocator(4))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:.1f}°E'))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:.1f}°N'))
    ax.tick_params(labelsize=6, length=2)

def footer_text(windows, uses_composite):
    t = 'Contains modified Copernicus Sentinel data (2020). Administrative boundaries: FAO GAUL 2015.'
    if uses_composite:
        t = (f"Temporal composites: R = Beginning ({window_text(windows['B'])}), "
             f"G = Middle ({window_text(windows['M'])}), B = Ending ({window_text(windows['E'])}).\n") + t
    return t

def make_figure(title, geom, fc, windows, panels, dim, out_png, rows=None):
    bb = bbox_of(geom)
    x0, y0, x1, y1 = bb
    imgs, errs = [], []
    for p in panels:
        try:
            imgs.append(fetch_png(panel_image(p, geom, fc, windows), geom, dim))
        except Exception as e:
            imgs.append(None); errs.append((p, str(e)[:200]))
    if all(im is None for im in imgs):
        raise RuntimeError(f'all panels failed: {errs}')
    n = len(panels)
    coslat = math.cos(math.radians((y0 + y1) / 2))
    ph = min(max(3.4 * (y1 - y0) / ((x1 - x0) * coslat), 2.0), 6.0)
    fig, axes = plt.subplots(1, n, figsize=(3.5 * n, ph + 1.2), squeeze=False)
    for k, (ax, p, im) in enumerate(zip(axes[0], panels, imgs)):
        if im is not None:
            ax.imshow(im, extent=[x0, x1, y0, y1], interpolation='nearest')
        else:
            ax.text((x0 + x1) / 2, (y0 + y1) / 2, 'image unavailable', ha='center', va='center', fontsize=7)
        format_axes(ax, bb)
        ax.set_title(f"({chr(97 + k)}) " + textwrap.fill(PANEL_TITLE[p], 34), fontsize=7)
        if p == 'index' and rows:
            for r in rows:
                if r.get('lon') is not None:
                    ax.text(r['lon'], r['lat'], r['district'], fontsize=LABEL_FONT, ha='center',
                            va='center', path_effects=HALO, zorder=4)
        if k == 0:
            draw_scalebar_north(ax, bb)
    fig.suptitle(title, fontsize=10, fontweight='bold')
    fig.text(0.01, 0.01, footer_text(windows, any(p not in ('index', 'truecolour') for p in panels)),
             fontsize=5.5, ha='left', va='bottom')
    fig.tight_layout(rect=[0, 0.07, 1, 0.93])
    os.makedirs(os.path.dirname(out_png), exist_ok=True)
    fig.savefig(out_png, dpi=DPI, facecolor='white')
    if SAVE_PDF:
        fig.savefig(out_png.replace('.png', '.pdf'), facecolor='white')
    plt.close(fig)
    return errs

def make_study_area(out_png):
    fills = None
    for st, col in zip(STATES_TO_RUN, STATE_COLOURS):
        lay = (ee.Image().byte().paint(featureCollection=STATE_FC[st], color=1)
               .visualize(min=0, max=1, palette=[col, col]))
        fills = lay if fills is None else fills.blend(lay)
    img = fills
    for st in STATES_TO_RUN:
        img = img.blend(outline_layer(STATE_FC[st], 1, 'ffffff'))
    for st in STATES_TO_RUN:
        state_outline = ee.FeatureCollection([ee.Feature(STATE_FC[st].geometry(100))])
        img = img.blend(outline_layer(state_outline, 2, '000000'))
    allgeom = ee.FeatureCollection([ee.Feature(STATE_FC[s].geometry(100)) for s in STATES_TO_RUN]).geometry(100)
    bb = bbox_of(allgeom)
    im = fetch_png(img, allgeom, 1800)
    x0, y0, x1, y1 = bb
    coslat = math.cos(math.radians((y0 + y1) / 2))
    fig, ax = plt.subplots(figsize=(6.5, min(max(6.5 * (y1 - y0) / ((x1 - x0) * coslat), 3), 8)))
    ax.imshow(im, extent=[x0, x1, y0, y1], interpolation='nearest')
    format_axes(ax, bb)
    for st in STATES_TO_RUN:
        c = STATE_FC[st].geometry(100).centroid(100).coordinates().getInfo()
        ax.text(c[0], c[1], st, fontsize=8, fontweight='bold', ha='center', va='center', path_effects=HALO, zorder=4)
    draw_scalebar_north(ax, bb)
    handles = [Patch(facecolor=col, edgecolor='black', label=f'{st} ({len(DIST_ROWS[st])} districts)')
               for st, col in zip(STATES_TO_RUN, STATE_COLOURS)]
    ax.legend(handles=handles, loc='lower right', fontsize=6, frameon=True)
    ax.set_title('Study area: Odisha, Chhattisgarh, Punjab and West Bengal (India)', fontsize=9, fontweight='bold')
    fig.text(0.01, 0.01, 'Administrative boundaries: FAO GAUL 2015.', fontsize=5.5, ha='left', va='bottom')
    fig.tight_layout(rect=[0, 0.03, 1, 1])
    os.makedirs(os.path.dirname(out_png), exist_ok=True)
    fig.savefig(out_png, dpi=DPI, facecolor='white')
    if SAVE_PDF:
        fig.savefig(out_png.replace('.png', '.pdf'), facecolor='white')
    plt.close(fig)

In [ ]:
# CELL 7 - District centroids (for the name labels) and Fig. 1 (study area)
for st in STATES_TO_RUN:
    cents = STATE_FC[st].map(lambda f: ee.Feature(None, {
        'lon': f.geometry().centroid(100).coordinates().get(0),
        'lat': f.geometry().centroid(100).coordinates().get(1)})).getInfo()['features']
    for row, c in zip(DIST_ROWS[st], cents):
        row['lon'], row['lat'] = c['properties']['lon'], c['properties']['lat']

FAILED = []
if MAKE_STUDY_AREA_FIGURE:
    path = os.path.join(OUT_DIR, '00_study_area.png')
    if not (SKIP_EXISTING and os.path.exists(path)):
        try:
            make_study_area(path)
            print('saved', path)
        except Exception as e:
            FAILED.append(('study area', str(e)[:200])); print('FAILED study area:', e)

saved figures_kharif2020/00_study_area.png


In [ ]:
# CELL 8 - One figure per state
if MAKE_STATE_FIGURES:
    for st in STATES_TO_RUN:
        safe_name = re.sub(r'[^A-Za-z0-9]+', '_', st)
        path = os.path.join(OUT_DIR, safe_name, f'state_{safe_name}.png')
        if SKIP_EXISTING and os.path.exists(path):
            print('exists, skipped:', path); continue
        t0 = time.time()
        try:
            errs = make_figure(f'{st}: {len(DIST_ROWS[st])} districts', STATE_FC[st].geometry(100), STATE_FC[st],
                               STATE_WINDOWS[st], PANELS_STATE, STATE_DIM, path, rows=DIST_ROWS[st])
            print(f'saved {path} ({time.time() - t0:.0f} s)', ('| panel errors: ' + str(errs)) if errs else '')
        except Exception as e:
            FAILED.append((st, str(e)[:200])); print('FAILED', st, e)

saved figures_kharif2020/Odisha/state_Odisha.png (156 s) 
saved figures_kharif2020/Chhattisgarh/state_Chhattisgarh.png (94 s) 
saved figures_kharif2020/Punjab/state_Punjab.png (196 s) 
saved figures_kharif2020/West_Bengal/state_West_Bengal.png (129 s) 


In [ ]:
# CELL 9 - One figure per district  (about 100+ figures; each needs 3 Earth Engine image requests,
if MAKE_DISTRICT_FIGURES:
    done = 0
    for st in STATES_TO_RUN:
        lst = STATE_FC[st].toList(STATE_FC[st].size())
        sname = re.sub(r'[^A-Za-z0-9]+', '_', st)
        for row in DIST_ROWS[st]:
            code_txt = str(int(row['code'])) if row['code'] is not None else f"{row['i']:03d}"
            dname = re.sub(r'[^A-Za-z0-9]+', '_', str(row['district'])).strip('_')
            path = os.path.join(OUT_DIR, sname, 'districts', f'{code_txt}_{dname}.png')
            if SKIP_EXISTING and os.path.exists(path):
                continue
            t0 = time.time()
            try:
                feat = ee.Feature(lst.get(row['i']))
                errs = make_figure(f"{st}: {row['district']} district ({row['area_km2']:,.0f} km²)",
                                   feat.geometry(), ee.FeatureCollection([feat]), STATE_WINDOWS[st],
                                   PANELS_DISTRICT, DISTRICT_DIM, path)
                done += 1
                print(f"saved {st} / {row['district']} ({time.time() - t0:.0f} s)", ('| panel errors: ' + str(errs)) if errs else '')
            except Exception as e:
                FAILED.append((st, row['district'], str(e)[:200]))
                print('FAILED', st, row['district'], str(e)[:120])
    print(f'\nDistrict figures made in this run: {done}')
print('Failed items (re-run the cell to retry them):', FAILED if FAILED else 'none')


District figures made in this run: 0
Failed items (re-run the cell to retry them): none


In [ ]:
# CELL 10 - Collect the figures (zip for download, optional copy to Google Drive)
zip_path = shutil.make_archive(OUT_DIR, 'zip', OUT_DIR)
print('Created', zip_path)

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    target = f'/content/drive/MyDrive/{DRIVE_FOLDER}_figures'
    shutil.copytree(OUT_DIR, target, dirs_exist_ok=True)
    print('Copied to', target)

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print('(Download manually from the Colab file browser.)')

Created /content/figures_kharif2020.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Notes for the paper
* **Figure caption / acknowledgement:** keep the footer "Contains modified Copernicus Sentinel data (2020)" (it is the Copernicus attribution requirement) and cite FAO GAUL for the boundaries. The thesis already notes the GAUL 2015 district layer predates newer districts.
* **Reading the composites:** in the NIR composite, rice is green (NIR peaks in the middle stage); in the VH composite, colour shows in which stage backscatter is highest. The explanation sits in the footer of each figure.
* **Output tree:** `figures_kharif2020/00_study_area.png`, `figures_kharif2020/<State>/state_<State>.png`, `figures_kharif2020/<State>/districts/<code>_<District>.png`.
* **Resolution:** the images are generated by Earth Engine at about 20-100 m per pixel depending on the area (not full 10 m). For full-resolution 10 m rasters use the export notebook.
* **If a figure fails** with a timeout, lower `STATE_DIM` / `DISTRICT_DIM` or run fewer panels, then re-run the cell.